# Packages import and installation

In [0]:
%pip install ..

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

from ag_vision.pipelines import annotation_processing as ap
from ag_vision.pipelines import dataframe_processing as dfp

# Variables set up

In [0]:
spark = SparkSession.builder.getOrCreate()

In [0]:
catalog_name = "use1_prod_artemis_catalog_3718194974443840"

In [0]:
# Annotations table name
table_name = dbutils.widgets.get("table_name")
TABLE_NAME = f"{catalog_name}.production.{table_name}"

In [0]:
# Set to True to generate the whole table from scratch.
# Set to False to only add new data (i.e., the latest uploaded annotations to production).
REGEN = dbutils.widgets.get("REGEN").strip().lower() == "true"
print(f"Regen is {REGEN}, if true it will recreate the whole table, if False it will append new data to the existing table.")

In [0]:
# Set the root directory variables
folder_name = dbutils.widgets.get("folder_name")
root_path = f"/Volumes/{catalog_name}/production/data/{folder_name}"

# Annotation data loading from source folder

In [0]:
# Load the root directory
#glob_pattern = f"{root_path}/annotations/*/*/*/*/*.{{json}}"
df = spark.read.format("binaryFile") \
    .option("recursiveFileLookup", "true") \
    .load(root_path) \
    .select("path")

# Filter by Depth AND Regex
# Your pattern: f"{root_path}/annotations/*/*/*/*/*.{{json}}
# This implies a specific number of folders between the root path and the file.
anno_paths_df = df.select("path").filter(
    # Ensure it's a json file
    F.col("path").rlike(r"\.(json)$") & 
    # Ensure 'annotations' folder is in the path
    F.col("path").contains("/annotations") &
    # Ensure the depth matches the 11-slash pattern.
    (F.size(F.split(F.col("path"), "/")) >= 11)
)

# Collect
anno_paths_list = [row.path.replace('dbfs:', '') for row in anno_paths_df.toLocalIterator()]
print(f"Found {len(anno_paths_list)} paths at the correct depth.")

In [0]:
old_df = spark.table(TABLE_NAME).select("annotation_path").toPandas()
print(f"The Old Table len is {len(old_df)}")

# Identification of newly uploaded annotations data

In [0]:
run_list = []

if not REGEN:
    s = pd.Series(anno_paths_list)
    run_list = s[~s.isin(old_df['annotation_path'])].tolist()
else:
    run_list = anno_paths_list
print(f"The len of the run list is. {len(run_list)}")

In [0]:
# Need to generate the schema.
if len(run_list) > 0:
    sample_df = ap.generate_annotations_table(
        anno_list=run_list[:1]
        )
    my_schema = spark.createDataFrame(sample_df).schema.simpleString()
    anno_item_spark_df = spark.createDataFrame([(i,) for i in run_list], ["item"])

In [0]:
if len(run_list) > 0:
    # Retrieve the existing images at the same rooth path as the annotations
    existing_images_sdf = (
        spark.read.format("binaryFile")
        .option("pathGlobFilter", "*.jpg")
        .option("recursiveFileLookup", "true")
        .load(root_path)
        .select(F.regexp_replace("path", "^dbfs:", "").alias("image_path"))
    )
    anno_spark_df = anno_item_spark_df.mapInPandas(
        dfp.process_annotations_batch,
        schema=my_schema
    )
    # left_semi: keeps annotation rows whose image exists, adds no columns
    valid_anno_spark_df = anno_spark_df.join(existing_images_sdf, "image_path", "left_semi")

In [0]:
files_sdf = spark.createDataFrame([(path,) for path in anno_paths_list], ["annotation_path"])

# Table update with new annotations data

In [0]:
# 1. Check if the table exists and REGEN is False
table_exists = spark.catalog.tableExists(TABLE_NAME)

if not REGEN and table_exists:
    # Create a DataFrame from anno_paths_list to avoid driver memory issues with .isin()
    old_spark_df = spark.read.table(TABLE_NAME).join(files_sdf, "annotation_path", "inner")
    
    # Combine (Union) the old data with the new results
    # unionByName is safer as it matches columns by name, not position
    if len(run_list) > 0:
        final_spark_df = old_spark_df.unionByName(valid_anno_spark_df)
    else:
        final_spark_df = old_spark_df
else:
    # If REGEN is True or table doesn't exist, just use the new data
    final_spark_df = valid_anno_spark_df

#2. Write the final result back to the table
final_spark_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABLE_NAME)